# Requirements

This notebook assumes the Oracle database already contains the ingested `research_papers` table, embeddings, and related search indexes.

It includes the setup needed to run the `Agent as Tools (Ochestration)` example.

In [1]:
import oracledb
import time

def connect_to_oracle(max_retries=3, retry_delay=5):
    """
    Connect to Oracle database with retry logic and better error handling.
    
    Args:
        max_retries: Maximum number of connection attempts
        retry_delay: Seconds to wait between retries
    """
    user = "system"
    password = "OraclePwd_2025"  # must match ORACLE_PWD from docker run
    dsn = "localhost:1521/FREEPDB1"
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(
                user=user,
                password=password,
                dsn=dsn
            )
            print("✓ Connected successfully!")
            
            # Test the connection
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%';")
                banner = cur.fetchone()[0]
                print(f"\n{banner}")
            
            return conn
            
        except oracledb.OperationalError as e:
            error_msg = str(e)
            print(f"✗ Connection failed (attempt {attempt}/{max_retries})")
            
            if "DPY-4011" in error_msg or "Connection reset by peer" in error_msg:
                print("  → This usually means:")
                print("    1. Database is still starting up (wait 2-3 minutes)")
                print("    2. Listener is not bound to 0.0.0.0 (run fix_oracle_listener())")
                print("    3. Container is not running (check with check_docker_container())")
                
                if attempt < max_retries:
                    print(f"\n  Waiting {retry_delay} seconds before retry...")
                    time.sleep(retry_delay)
                else:
                    print("\n  💡 Try running:")
                    print("     1. check_docker_container() - verify container is running")
                    print("     2. fix_oracle_listener() - fix listener binding")
                    raise
            else:
                raise
        except Exception as e:
            print(f"✗ Unexpected error: {e}")
            raise
    
    raise ConnectionError("Failed to connect after all retries")

# Connect to Oracle
conn = connect_to_oracle()

Connection attempt 1/3...
✓ Connected successfully!

Oracle AI Database 26ai Free Release 23.26.1.0.0 - Develop, Learn, and Run for Free


In [2]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

<All keys matched successfully>


In [3]:
import numpy as np
import array

In [4]:
def keyword_search_research_papers(conn, keyword: str):
    """
    Perform a full-text keyword search on the 'text' column 
    using the Oracle Text index (rp_text_idx).

    Args:
        conn: Oracle database connection object.
        keyword (str): Keyword or phrase to search for.

    Returns:
        tuple: (rows, columns)
    """
    query = """
        SELECT 
            arxiv_id, 
            title, 
            SUBSTR(text, 1, 200) AS text_snippet,
            SCORE(1) AS relevance_score
        FROM research_papers
        WHERE CONTAINS(text, :keyword, 1) > 0
        ORDER BY SCORE(1) DESC
        FETCH FIRST 10 ROWS ONLY
    """

    with conn.cursor() as cur:
        cur.execute(query, keyword=keyword)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    return rows, columns


In [5]:
SEARCH_QUERY = "Get me papers related to planetary exploration"

In [6]:
def vector_search_research_papers(conn, embedding_model, search_query: str, top_k: int = 5):
    """
    Perform a vector similarity search on the research_papers table using a query embedding.
    Returns cosine similarity scores (higher = more similar).
    """

    # 1️⃣ Encode the query into a vector
    query_embedding = embedding_model.encode(
        [f"search_query: {search_query}"], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()

    # 2️⃣ Prepare the vector for Oracle binding
    query_embedding_array = array.array('f', query_embedding)

    # 3️⃣ Run a vector similarity search using cosine similarity
    query = f"""
        SELECT 
            arxiv_id, 
            title, 
            abstract, 
            SUBSTR(text, 1, 200) AS text_snippet,
            ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
        FROM research_papers
        ORDER BY similarity_score DESC
        FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
    """

    # 4️⃣ Execute and return results
    with conn.cursor() as cur:
        cur.execute(query, q=query_embedding_array)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    return rows, columns


In [7]:
import array
import numpy as np

def hybrid_search_research_papers_pre_filter(
    conn,
    embedding_model,
    search_phrase: str,
    top_k: int = 10,
    show_explain: bool = False
):
    """
    Perform a hybrid search using Oracle Text + Vector Search.
    Combines lexical filtering (CONTAINS) with semantic re-ranking via cosine similarity.

    Args:
        conn: Oracle database connection object.
        embedding_model: Model with `.encode()` method (e.g., SentenceTransformer).
        search_phrase (str): User search phrase used for both text filtering and embedding.
        top_k (int): Number of results to return (default = 10).
        show_explain (bool): If True, prints the execution plan.

    Returns:
        tuple: (rows, columns, exec_plan_text or None)
    """

    # --- Step 1: Encode search phrase into normalized vector ---
    query_embedding = embedding_model.encode(
        [f"search_query: {search_phrase}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    query_embedding_array = array.array('f', query_embedding)

    with conn.cursor() as cur:
        # Enable runtime stats if needed
        if show_explain:
            cur.execute("ALTER SESSION SET statistics_level = ALL")

        # --- Step 2: Hybrid query (Oracle Text + Vector) ---
        sql = f"""
            SELECT {"/*+ GATHER_PLAN_STATISTICS */" if show_explain else ""}
                arxiv_id,
                title,
                abstract,
                SUBSTR(text, 1, 200) AS text_snippet,
                ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
            FROM research_papers
            WHERE CONTAINS(text, :kw, 1) > 0
            ORDER BY similarity_score DESC
            FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
        """

        cur.execute(sql, q=query_embedding_array, kw=search_phrase)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    # --- Step 3: Execution plan (optional) ---
    exec_plan_text = None
    if show_explain:
        with conn.cursor() as cur_plan:
            cur_plan.execute("""
                SELECT plan_table_output
                FROM TABLE(DBMS_XPLAN.DISPLAY_CURSOR(NULL, NULL, 'ALLSTATS LAST +PREDICATE'))
            """)
            exec_plan_text = "\n".join(r[0] for r in cur_plan.fetchall())

        print("\n====== Execution Plan (DBMS_XPLAN.DISPLAY_CURSOR) ======")
        print(exec_plan_text)
        print("========================================================\n")

    return rows, columns, exec_plan_text


In [8]:
# Azure OpenAI environment setup helpers
import getpass
import os

# Function to securely get and set environment variables
def set_env_securely_azure(var_name, prompt):
    value = getpass.getpass(prompt)
    os.environ[var_name] = value

In [9]:
# https://azure-agent-ai-foundry-resource.openai.azure.com/
# gpt-4o
# gpt-4.1
# gpt-5

set_env_securely_azure("AZURE_OPENAI_ENDPOINT", "Enter your Azure OpenAI endpoint (e.g. https://<resource>.openai.azure.com): ")
set_env_securely_azure("AZURE_OPENAI_DEPLOYMENT", "Enter your Azure OpenAI deployment name (e.g. gpt-4o): ")
print("RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.")

RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.


In [10]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM RESEARCH_PAPERS")
    print("Row count:", cur.fetchone()[0])

    cur.execute("""
        SELECT arxiv_id, title, abstract, text FROM RESEARCH_PAPERS
        FETCH FIRST 3 ROWS ONLY
    """)
    for row in cur.fetchall():
        print(row)

Row count: 1000
('0902.0428', 'Dynamics of planets in retrograde mean motion resonance', 'In a previous paper (Gayon &amp; Bois 2008a), we have shown the general efficiency of retrograde resonances for stabilizing compact planetary systems. Such retrograde resonances can be found when two-planets of a three-body planetary system are both in mean motion resonance and revolve in opposite directions. For a particular two-planet system, we have also obtained a new orbital fit involving such a counter-revolving configuration and consistent with the observational data. <br>In the present paper, we analytically investigate the three-body problem in this particular case of retrograde resonances. We therefore define a new set of canonical variables allowing to express correctly the resonance angles and obtain the Hamiltonian of a system harboring planets revolving in opposite directions. The acquiring of an analytical &#34;rail&#34; may notably contribute to a deeper understanding of our numeri

In [11]:
from agents import Agent, Runner

In [12]:
# Azure companion: configure openai-agents to use Azure OpenAI via DefaultAzureCredential
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AsyncAzureOpenAI
from agents import Agent, set_default_openai_client, set_tracing_disabled

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

# Normalize endpoint in case env var includes /openai or deployment path.
raw_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
if "/openai" in raw_endpoint.lower():
    raw_endpoint = raw_endpoint[: raw_endpoint.lower().index("/openai")]

AZURE_OPENAI_MODEL = os.environ["AZURE_OPENAI_DEPLOYMENT"]
# Runner uses Responses API; 2024-10-21 commonly fails on /responses in Azure.
env_api_version = os.environ.get("AZURE_OPENAI_API_VERSION")
if not env_api_version or env_api_version == "2024-10-21":
    AZURE_OPENAI_API_VERSION = "2025-03-01-preview"
else:
    AZURE_OPENAI_API_VERSION = env_api_version

azure_agents_client = AsyncAzureOpenAI(
    azure_endpoint=raw_endpoint,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_ad_token_provider=token_provider,
    # Compatibility for openai<2.x credential gate when using only AAD token provider.
    _enforce_credentials=False,
)

# Guard: fail fast if this ever gets replaced with a sync client
if not isinstance(azure_agents_client, AsyncAzureOpenAI):
    raise TypeError(
        "Expected AsyncAzureOpenAI for agent runs. Restart kernel and rerun this cell before Azure runs."
    )

print(type(azure_agents_client))
print(f"Azure endpoint: {raw_endpoint}")
print(f"Azure deployment: {AZURE_OPENAI_MODEL}")
print(f"Azure API version: {AZURE_OPENAI_API_VERSION}")

# Route Agent/Runner calls to Azure OpenAI for the Azure companion cells
set_default_openai_client(azure_agents_client)
set_tracing_disabled(disabled=True)

research_paper_assistant_azure = Agent(
    name="Research Paper Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="""
      You are a Research Paper Assistant focused on helping users explore, analyze, and summarize
      academic research.

      Maintain a professional, concise, and scholarly tone appropriate for research discussions.
    """,
)

<class 'openai.lib.azure.AsyncAzureOpenAI'>
Azure endpoint: https://azure-agent-ai-foundry-resource.openai.azure.com
Azure deployment: gpt-4o
Azure API version: 2025-03-01-preview


In [13]:
from agents.tool import function_tool

@function_tool
def get_research_papers(user_query: str, retrieval_mode: str = "hybrid", top_k: int = 5) -> str:
    """
    Retrieves academic research papers relevant to the user's query.

    This tool queries the research_papers SQL table using one of three retrieval techniques:
        - 'keyword'  → lexical search via LIKE filtering
        - 'vector'   → semantic similarity search
        - 'hybrid'   → combines keyword prefiltering + vector similarity (default)

    Use this tool when analyzing or summarizing scientific literature.

    Args:
        user_query (str): Research topic or question to search for.
        retrieval_mode (str): 'keyword', 'vector', or 'hybrid'. Default is 'hybrid'.
        top_k (int): Number of top papers to retrieve (default=5).

    Returns:
        str: A formatted summary of the most relevant research papers.
    """

    # ------------------------------------------------------------------
    # Perform retrieval using SQL-based functions (defined earlier)
    # ------------------------------------------------------------------
    if retrieval_mode == "keyword":
        rows, columns = keyword_search_research_papers(conn, user_query)
    elif retrieval_mode == "vector":
        rows, columns = vector_search_research_papers(conn, embedding_model, user_query, top_k)
    else:
        rows, columns, _ = hybrid_search_research_papers_pre_filter(
            conn=conn,
            embedding_model=embedding_model,
            search_phrase=user_query,
            top_k=top_k,
            show_explain=False
        )

    retrieved_count = len(rows) if rows else 0

    # ------------------------------------------------------------------
    # Format the output into a readable string
    # ------------------------------------------------------------------
    if retrieved_count == 0:
        return f"No research papers found related to '{user_query}'."

    formatted_results = [f"📚 {retrieved_count} papers retrieved for query: '{user_query}'\n"]
    for i, row in enumerate(rows):
        row_data = dict(zip(columns, row))
        title = row_data.get("TITLE", "Untitled Paper")
        abstract = row_data.get("ABSTRACT", "No abstract available.")
        score = (
            row_data.get("SIMILARITY_SCORE")
            or row_data.get("TEXT_RELEVANCE_SCORE")
            or "N/A"
        )
        formatted_results.append(
            f"[{i+1}] {title}\n"
            f"Abstract: {abstract}\n"
            f"Relevance Score: {score}\n"
        )

    return "\n".join(formatted_results)


In [14]:
from agents.tool import function_tool

@function_tool
def get_past_research_conversations(user_query: str, top_k: int = 5) -> str:
    """
    Retrieves relevant past research-related conversations or analyses related to the query.

    This tool searches a SQL database of prior research assistant conversations, 
    literature discussions, or synthesis sessions to find relevant context. 
    It allows the research assistant to recall previous analyses or summaries 
    that addressed similar topics, providing continuity and richer insights.

    Args:
        user_query (str): The research topic, concept, or question to search for.
        top_k (int): Number of top past discussions to retrieve (default=5).

    Returns:
        str: Formatted examples of relevant past research discussions.
    """

    # ------------------------------------------------------------------
    # Perform retrieval using the SQL-based hybrid search (vector + keyword)
    # ------------------------------------------------------------------
    rows, columns, _ = hybrid_search_research_papers_pre_filter(
        conn=conn,
        embedding_model=embedding_model,
        search_phrase=user_query,
        top_k=top_k,
        show_explain=False
    )

    retrieved_count = len(rows) if rows else 0

    # ------------------------------------------------------------------
    # Format results for readability
    # ------------------------------------------------------------------
    if retrieved_count == 0:
        return f"No past research discussions found related to '{user_query}'."

    formatted_results = [f"🧠 {retrieved_count} past research discussions retrieved for query: '{user_query}'\n"]
    for i, row in enumerate(rows):
        row_data = dict(zip(columns, row))
        title = row_data.get("TITLE", "Untitled Discussion")
        abstract = row_data.get("ABSTRACT", "No summary available.")
        snippet = row_data.get("TEXT_SNIPPET", "")
        score = (
            row_data.get("SIMILARITY_SCORE")
            or row_data.get("TEXT_RELEVANCE_SCORE")
            or "N/A"
        )
        formatted_results.append(
            f"[{i+1}] **{title}**\n"
            f"Summary: {abstract}\n"
            f"Snippet: {snippet}\n"
            f"Relevance Score: {score}\n"
        )

    return "\n".join(formatted_results)


## Agent as Tools (Ochestration)


Add an image of the flow of these agents

In [ ]:
# # Define specialized agents for different research retrieval tasks
# research_paper_agent = Agent(
#     name="research_paper_agent",
#     instructions="""
#         You specialize in retrieving and summarizing academic research papers.
#         Use the get_research_papers tool to find relevant literature based on the user's query.
#         Always cite sources using [1], [2], etc., and focus on summarizing key findings,
#         methodologies, and implications of the studies retrieved.
#     """,
#     handoff_description="A research retrieval specialist with access to academic papers and literature databases.",
#     tools=[get_research_papers],
# )

# research_conversation_agent = Agent(
#     name="research_conversation_agent",
#     instructions="""
#         You specialize in retrieving and summarizing past research discussions and analyses.
#         Use the get_past_research_conversations tool to surface relevant prior sessions
#         or summaries that relate to the user's current topic of inquiry.
#         Present these as context and examples of prior analytical reasoning.
#     """,
#     handoff_description="A research memory specialist with access to prior academic discussions and analyses.",
#     tools=[get_past_research_conversations],
# )


In [15]:
# Azure companion: specialized agents with Azure deployment
research_paper_agent_azure = Agent(
    name="research_paper_agent_azure",
    model=AZURE_OPENAI_MODEL,
    instructions="""
        You specialize in retrieving and summarizing academic research papers.
        Use the get_research_papers tool to find relevant literature based on the user's query.
        Always cite sources using [1], [2], etc., and focus on summarizing key findings,
        methodologies, and implications of the studies retrieved.
    """,
    handoff_description="A research retrieval specialist with access to academic papers and literature databases.",
    tools=[get_research_papers],
)

research_conversation_agent_azure = Agent(
    name="research_conversation_agent_azure",
    model=AZURE_OPENAI_MODEL,
    instructions="""
        You specialize in retrieving and summarizing past research discussions and analyses.
        Use the get_past_research_conversations tool to surface relevant prior sessions
        or summaries that relate to the user's current topic of inquiry.
        Present these as context and examples of prior analytical reasoning.
    """,
    handoff_description="A research memory specialist with access to prior academic discussions and analyses.",
    tools=[get_past_research_conversations],
)

In [ ]:
# # Create an orchestrator agent that can coordinate both research retrieval agents
# orchestrator_agent = Agent(
#     name="research_assistant_orchestrator",
#     instructions=(
#         "You are a Research Orchestrator Assistant responsible for coordinating information retrieval "
#         "across multiple specialized research tools.\n\n"
#         "Your role is to help users explore, analyze, and synthesize academic research efficiently.\n\n"
#         "IMPORTANT RULES:\n"
#         "1. ALWAYS use translate_to_research_papers when a query mentions research papers, studies, or findings.\n"
#         "2. ALWAYS use translate_to_research_conversations when a query mentions previous discussions, analyses, or summaries.\n"
#         "3. If a query requests BOTH new research and past discussions, use BOTH tools in sequence.\n"
#         "4. NEVER attempt to provide research summaries without using your tools.\n"
#         "5. Each tool provides complementary context — use all appropriate tools for a comprehensive academic response.\n\n"
#         "After retrieving relevant results, synthesize them into a cohesive summary:\n"
#         "- Clearly distinguish between newly retrieved research and recalled past discussions.\n"
#         "- Cite sources using [1], [2], etc.\n"
#         "- Identify key insights, trends, and research gaps.\n"
#         "- Maintain an academic and objective tone."
#     ),
#     tools=[
#         research_paper_agent.as_tool(
#             tool_name="translate_to_research_papers",
#             tool_description="Retrieve and summarize relevant academic research papers and literature findings.",
#         ),
#         research_conversation_agent.as_tool(
#             tool_name="translate_to_research_conversations",
#             tool_description="Retrieve and summarize past research discussions or analyses related to the topic.",
#         ),
#     ],
# )


In [16]:
# Azure companion orchestrator
orchestrator_agent_azure = Agent(
    name="research_assistant_orchestrator_azure",
    model=AZURE_OPENAI_MODEL,
    instructions=(
        "You are a Research Orchestrator Assistant responsible for coordinating information retrieval "
        "across multiple specialized research tools.\n\n"
        "Your role is to help users explore, analyze, and synthesize academic research efficiently.\n\n"
        "IMPORTANT RULES:\n"
        "1. ALWAYS use translate_to_research_papers when a query mentions research papers, studies, or findings.\n"
        "2. ALWAYS use translate_to_research_conversations when a query mentions previous discussions, analyses, or summaries.\n"
        "3. If a query requests BOTH new research and past discussions, use BOTH tools in sequence.\n"
        "4. NEVER attempt to provide research summaries without using your tools.\n"
        "5. Each tool provides complementary context — use all appropriate tools for a comprehensive academic response.\n\n"
        "After retrieving relevant results, synthesize them into a cohesive summary:\n"
        "- Clearly distinguish between newly retrieved research and recalled past discussions.\n"
        "- Cite sources using [1], [2], etc.\n"
        "- Identify key insights, trends, and research gaps.\n"
        "- Maintain an academic and objective tone."
    ),
    tools=[
        research_paper_agent_azure.as_tool(
            tool_name="translate_to_research_papers",
            tool_description="Retrieve and summarize relevant academic research papers and literature findings.",
        ),
        research_conversation_agent_azure.as_tool(
            tool_name="translate_to_research_conversations",
            tool_description="Retrieve and summarize past research discussions or analyses related to the topic.",
        ),
    ],
)

In [ ]:
# # Final agent to synthesize information from all sources (Research use case)
# synthesizer_agent = Agent(
#     name="research_response_synthesizer",
#     instructions=(
#         "You create comprehensive, well-organized research summaries by combining information from multiple sources.\n\n"
#         "When organizing your response:\n"
#         "1) Start with a concise abstract-style overview (3–5 sentences) highlighting key findings and takeaways.\n"
#         "2) Clearly separate NEW LITERATURE FINDINGS from PAST RESEARCH DISCUSSIONS.\n"
#         "3) Cite sources using bracketed numbers [1], [2], etc., aligned with the retrieved items.\n"
#         "4) Emphasize methods, evidence strength, and limitations; avoid speculation beyond the provided context.\n"
#         "5) Use clear, scannable formatting (short paragraphs, bullet points where appropriate).\n"
#         "6) Conclude with open questions, gaps, or future work suggested by the literature.\n"
#         "7) If evidence is sparse, state this explicitly and avoid overgeneralization.\n"
#         "Tone: academic, objective, and precise."
#     ),
# )


In [17]:
# Azure companion synthesizer
synthesizer_agent_azure = Agent(
    name="research_response_synthesizer_azure",
    model=AZURE_OPENAI_MODEL,
    instructions=(
        "You create comprehensive, well-organized research summaries by combining information from multiple sources.\n\n"
        "When organizing your response:\n"
        "1) Start with a concise abstract-style overview (3–5 sentences) highlighting key findings and takeaways.\n"
        "2) Clearly separate NEW LITERATURE FINDINGS from PAST RESEARCH DISCUSSIONS.\n"
        "3) Cite sources using bracketed numbers [1], [2], etc., aligned with the retrieved items.\n"
        "4) Emphasize methods, evidence strength, and limitations; avoid speculation beyond the provided context.\n"
        "5) Use clear, scannable formatting (short paragraphs, bullet points where appropriate).\n"
        "6) Conclude with open questions, gaps, or future work suggested by the literature.\n"
        "7) If evidence is sparse, state this explicitly and avoid overgeneralization.\n"
        "Tone: academic, objective, and precise."
    ),
)

In [18]:
from agents import ItemHelpers, MessageOutputItem, trace
from agents import Runner  # assuming Runner is imported elsewhere; include here for clarity


In [ ]:
# async def research_assistant_workflow(user_query: str):
#     """Run the complete research assistant workflow (orchestrate retrieval + synthesize)."""
#     # 1) Have the research orchestrator decide which tools to invoke
#     with trace("Research Orchestrator"):
#         orchestrator_result = await Runner.run(orchestrator_agent, user_query)

#         # Debug/transparency: print intermediate orchestration steps
#         print("\n--- Research Orchestration Steps ---")
#         for item in orchestrator_result.new_items:
#             if isinstance(item, MessageOutputItem):
#                 text = ItemHelpers.text_message_output(item)
#                 if text:
#                     print(f"  - Retrieval step: {text}")

#         # 2) Synthesize all gathered information into a cohesive research summary
#         synthesizer_result = await Runner.run(
#             synthesizer_agent, orchestrator_result.to_input_list()
#         )

#         print(f"\n\n--- Final Research Synthesis ---\n{synthesizer_result.final_output}\n")

#     return synthesizer_result.final_output


In [19]:
async def research_assistant_workflow_azure(user_query: str):
    """Run the complete Azure research assistant workflow (orchestrate retrieval + synthesize)."""
    with trace("Research Orchestrator Azure"):
        orchestrator_result = await Runner.run(orchestrator_agent_azure, user_query)

        print("\n--- Research Orchestration Steps (Azure) ---")
        for item in orchestrator_result.new_items:
            if isinstance(item, MessageOutputItem):
                text = ItemHelpers.text_message_output(item)
                if text:
                    print(f"  - Retrieval step: {text}")

        synthesizer_result = await Runner.run(
            synthesizer_agent_azure, orchestrator_result.to_input_list()
        )

        print(f"\n\n--- Final Research Synthesis (Azure) ---\n{synthesizer_result.final_output}\n")

    return synthesizer_result.final_output

In [20]:
import asyncio
import nest_asyncio

# Apply nest_asyncio to patch the event loop
nest_asyncio.apply()

In [ ]:
# def run_virtual_research_assistant(query):
#     # Create a new event loop
#     loop = asyncio.new_event_loop()
#     asyncio.set_event_loop(loop)

#     # Run the async function and get the result
#     result = loop.run_until_complete(research_assistant_workflow(query))

#     # Clean up
#     loop.close()

#     return result

In [21]:
def run_virtual_research_assistant_azure(query):
    # Create a new event loop
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    # Run the async function and get the result
    result = loop.run_until_complete(research_assistant_workflow_azure(query))

    # Clean up
    loop.close()

    return result

In [ ]:
# # Now call the function this way
# query = input("What research topic can I help you with today? ")
# run_virtual_research_assistant(query)

In [22]:
# mission planning

# Azure companion invocation
query_azure = input("[Azure] What research topic can I help you with today? ")
run_virtual_research_assistant_azure(query_azure)


--- Research Orchestration Steps (Azure) ---
  - Retrieval step: ### Synthesis: Research on Mission Planning

#### New Research Insights
A recent study focused on the **EPOXI mission’s planning and execution** targeting comet 103P/Hartley 2 [1]. Key findings from the study include:
- **Precise Astrometry**: Researchers confirmed the comet's structural integrity and position, ensuring accurate targeting for the spacecraft.
- **Nucleus Characterization**: The comet’s nucleus was estimated at ≤1 km in radius with a surface albedo of ~6%.
- **Activity Observations**: Faint activity around the nucleus, likely dust trails from past outbursts, was detected and analyzed.
These efforts significantly informed the mission's trajectory and observational plans, demonstrating the importance of initial scouting data for spacecraft navigation.

#### Recalled Discussions
Previous discussions on mission planning for the same EPOXI mission emphasized the importance of:
- **Early Detection**: Tracking an

"### Synthesis: Research on Mission Planning for Spacecraft Missions\n\n#### New Research Insights\nA recent study addressed mission planning challenges for the **EPOXI mission**, which targeted the comet 103P/Hartley 2 [1]. Key findings include:\n- **Astrometry for Navigation**: Researchers used precise astrometric data to confirm the comet’s position and ensure it matched the predicted trajectory, which was critical for accurate targeting.\n- **Nucleus Properties and Activity**: The nucleus was confirmed to have a radius of ≤1 km and an albedo of ~6%. Sparse activity around the nucleus, likely from residual dust trails, was noted.\n- **Mission Adjustments**: Observations guided strategic monitoring and influenced mission trajectory design, contributing to the overall success of the spacecraft’s interaction with the comet.\n\n### Key Takeaways for Mission Planning\n1. **Early Observations Are Crucial**: Confirming structural integrity, position, and physical parameters of the target h